# 42. `max_bin`, the one setting nothing in this ledger has ever touched

**One variable against ledger row 93** (`xgb_te_fe`, CV 0.968005, the best single model here):
XGBoost's `max_bin`. Same 49 features, same encoder fingerprinted at `0642e41750ef8bab`, same
ratio block, same folds, same seed, same budget, same depth.

## Why this is not a fifth knob sweep

Histogram boosters do not split on raw values. They bucket each column first, and only bucket
edges become candidate split points. **A tree cannot see inside a bucket.**

Measured on this competition's own data, train and test together:

| column | distinct values |
|---|---|
| `weekend_screen_time` | **1,459** |
| `daily_screen_time_hours` | **1,397** |
| `social_media_hours` | 729 |
| `work_study_hours` | 601 |
| everything else | 451 or fewer |

And the defaults every model in this repo has been running on:

| library | setting | default |
|---|---|---|
| LightGBM | `max_bin` | 255 |
| XGBoost | `max_bin` | 256 |
| CatBoost | `border_count` | 254 |

So the two highest-cardinality columns are being compressed by roughly **5.7 to 1**, and about
six adjacent distinct values share a bucket. **Nothing in 93 ledger rows has ever set this.**

This is a knob, and the six-point rule in `NOTES.md` assigns knobs a low prior. It went seven for
seven before the CatBoost budget broke the streak. The distinction that saved that one applies
here and is stronger: this is not a capacity parameter, it is a **resolution** parameter, and
there is an independent measurement attached rather than a mechanism invented for the occasion.

## The evidence that this repo already had and misread

Row 19, the decimal lattice, measured the target rate by first decimal digit on the full training
set and found effects far outside noise:

| column | swing across digits | vs its own standard error |
|---|---|---|
| `weekend_screen_time` | 11.54 pts | ~60 se |
| `daily_screen_time_hours` | 8.90 pts | ~47 se |

Row 19 concluded the digit was worth nothing incrementally, at -0.000132, and `NOTES.md`
explained it as *the digit is a deterministic function of the value, and target encoding the
exact value already carries it.*

**That explanation has a hole in it that nobody checked.** Target encoding `weekend_screen_time`
produces `te_weekend_screen_time`, which has as many distinct values as the column it came from,
about 1,459. That column is then **handed to a histogram booster that buckets it to 256**. The
per-value offsets the encoder faithfully computed are averaged back together downstream. The
encoder was never the thing throwing away the resolution; the binner was, and it did it to the
encoding as well as to the raw column.

If that reading is right, row 19's effect is real, is still in the data, and has never actually
reached a model.

## The public measurement

kito_pl swept this on 2026-08-22 and reports **+0.0023 OOF** from `max_bin` alone, larger than 26
Optuna trials (+0.0010) and every feature idea they tried. Their curve rises from 255 to 1439 and
then stops dead, and 1439 is their largest distinct count. They moved the cause to check it:
rounding the numerics to one decimal cut the largest distinct count to 231 and the stopping point
moved to 255. **91 percent of the gain came from the two columns with more than 1,400 distinct
values**, which are the same two named above.

Their fold split is the frozen community one, already verified bit-identical to ours, so their
number is comparable rather than merely suggestive.

## The grid, pre-registered

`256, 512, 1024, 1536, 3072`.

`256` is XGBoost's default and therefore row 93's value, so that arm is the reproduction check.
The grid brackets the 1,459 distinct-value ceiling on both sides, so if kito_pl's rule holds the
curve should flatten between 1536 and 3072 rather than continuing.

## What the smoke run changed about this, recorded because it moved the prediction

The first draft predicted **under** kito_pl's +0.0023, on the argument that our encoder and ratio
block already recover per-value structure by other means. The smoke run printed the frame the
model actually sees and that argument is at best half right. **27 of 49 columns carry more than
256 distinct values**, and the worst offenders are the ratio columns added in row 93:

| column | distinct values on the model's frame |
|---|---|
| `screen_to_sleep` | 12,108 |
| `work_share` | 11,789 |
| `notif_per_hour` | 11,104 |
| ... eight more ratio columns | 8,029 to 10,814 |
| `weekend_screen_time` | 1,168 |
| `te_notifications_per_day` | 839 |

So the ratio block measured at +0.000906 in row 93 was achieved while being compressed roughly
**47 to 1**, and the target encodings are compressed too, which is the same point the header makes
about row 19.

**That cuts both ways and the notebook should not pretend otherwise.** kito_pl's mechanism is
about the generator's per-value offsets, which live in the *original* columns and are inherited by
their encodings. A ratio's distinct values are arithmetic consequences of two columns; there is no
lookup table behind them, so finer bins there should buy little for that reason even though the
cardinality is enormous. If that is right, the ceiling that matters is the **1,459** of the raw
columns, not the 12,108 of `screen_to_sleep`, and the grid to 3072 brackets it comfortably.

If the curve is **still climbing at 3072**, that reasoning is wrong, the ratio columns want bins
too, and the grid needs extending. That is a pre-registered outcome rather than a post-hoc escape.

## The prediction, written before the run

**+0.0010 to +0.0030, flattening between 1536 and 3072, landing between 0.9690 and 0.9710.**

The band is wider at the top than the first draft because of the cardinality table above.

**My last two predictions were wrong**, in opposite directions: row 39's magnitude by fourfold,
row 40's pattern backwards. Both were mechanism arguments I invented for the occasion. This one is
different in the way the CatBoost budget was different, and that is the only reason to trust it
more: it has an independent measurement on the same fold split, plus a prior measurement inside
this repo (row 19) that the mechanism explains and that nothing else has explained.

**The honest case against.** If our encoder and ratio block already recover what the bins are
throwing away, the curve is flat and this is the fifth knob null. That would also mean row 19's
explanation was right for the wrong reason and should be left alone.

## What this decides

Nothing about the stack. It writes one member vector per arm. Membership is a separate notebook
and a separate ledger row. No submission csv.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Row 93's configuration, held fixed. Only max_bin is swept.
LR = 0.05
N_EST = 2000
MAX_DEPTH = 6
BENCH_EST = 200
PROBE_FOLD = 0
THREADS = -1

# The knob. 256 is XGBoost's default and row 93's value, so that arm is the reproduction
# check. The grid brackets the 1,459 largest distinct-value count on both sides.
MAX_BINS = [256, 512, 1024, 1536, 3072]
BASE_BIN = 256

BASELINE_NAME, BASELINE_CV, BASELINE_ROW = "xgb_te_fe", 0.968005, "row 93 xgb_te_fe"
MAX_HOURS = 9.0
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   max_bin grid {MAX_BINS}   base {BASE_BIN}")

## Stage 1. Data, folds, leak checklist

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")
print(f"xgboost {xgb.__version__}, pandas {pd.__version__}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# The premise of the whole notebook, measured here rather than quoted from the header.
_both = pd.concat([train_full[COLS], test[COLS]], ignore_index=True)
DISTINCT = {c: int(_both[c].nunique(dropna=True)) for c in COLS}
MAX_DISTINCT = max(DISTINCT.values())
print(f"\ndistinct values per column, train and test together:")
for c, n in sorted(DISTINCT.items(), key=lambda kv: -kv[1]):
    print(f"  {c:<26}{n:>8,}")
print(f"largest distinct count {MAX_DISTINCT:,}, so the grid must reach past it")
del _both
gc.collect()

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
X = train[COLS].copy()
X_test = test[COLS].copy()

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else "fold alignment: MISMATCH")

## Stage 2. The encoder and the ratio block, both copied from row 93

The encoder is fingerprinted against `13_target_encoding.ipynb`. The ratio block is the same 13
columns notebook `40` defined, and `missing_count` stays excluded for the reason `04` gives.

In [ ]:
X = train[COLS].copy()
X_test = test[COLS].copy()

def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

In [ ]:
DST, SM, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                   "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    """Notebook 40's 13 composition features. A pure function of the feature columns."""
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
train_perm = train.copy()
train_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(train_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del train_perm
gc.collect()


def build_arm(fold, want_test=False):
    """Row 93's frame: 36 encoded columns plus the 13 raw ratio columns."""
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test if want_test else None)
    for c in CAT:
        Xtr[c] = Xtr[c].astype("category")
        Xva[c] = Xva[c].astype("category")
    Xtr = pd.concat([Xtr, ratio_block(train.iloc[tr]).reset_index(drop=True)], axis=1)
    Xva = pd.concat([Xva, ratio_block(train.iloc[va]).reset_index(drop=True)], axis=1)
    if want_test:
        for c in CAT:
            Xte[c] = Xte[c].astype("category")
        Xte = pd.concat([Xte, ratio_block(test).reset_index(drop=True)], axis=1)
    return tr, va, Xtr, Xva, Xte


# How much resolution is actually at stake, on the frame the model sees. The encoded
# columns inherit the cardinality of the column they encode, which is the part row 19's
# explanation missed.
_tr, _va, _Xtr, _Xva, _ = build_arm(PROBE_FOLD)
nun = _Xtr.nunique()
print(f"\nfeatures on the model's frame: {_Xtr.shape[1]}")
print(f"columns with more distinct values than the default 256 bins:")
for c, n in nun[nun > 256].sort_values(ascending=False).items():
    print(f"  {c:<30}{n:>8,}")
print(f"  ({int((nun > 256).sum())} of {len(nun)} columns are being compressed at max_bin=256)")
del _Xtr, _Xva
gc.collect()

## Stage 3. Bench and the projection

In [ ]:
def make_xgb(n_est, max_bin):
    # Row 93's configuration, with max_bin as the only thing that varies.
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True, max_bin=max_bin,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=THREADS, verbosity=0,
    )


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


tr0, va0, Xtr0, Xva0, _ = build_arm(PROBE_FOLD)
bench = {}
for mb in MAX_BINS:
    t0 = time.time()
    m = make_xgb(BENCH_EST, mb)
    m.fit(Xtr0, y[tr0])
    secs = time.time() - t0
    p1 = m.predict_proba(Xva0)[:, 1]
    m2 = make_xgb(BENCH_EST, mb)
    m2.fit(Xtr0, y[tr0])
    drift = float(np.max(np.abs(p1 - m2.predict_proba(Xva0)[:, 1])))
    bench[mb] = secs
    print(f"max_bin {mb:>5}: bench AUC {roc_auc_score(y[va0], p1):.6f} in {hhmm(secs)}, "
          f"drift {drift:.3e} {'OK' if drift == 0.0 else 'NOT DETERMINISTIC'}")
    del m, m2
    gc.collect()
del Xtr0, Xva0
gc.collect()

projected = sum(bench[mb] / BENCH_EST * N_EST * 5 for mb in MAX_BINS)
print(f"\nprojected full run: {hhmm(projected)} for {len(MAX_BINS)} arms x 5 folds")
GO = projected < MAX_HOURS * 3600 or SMOKE
print("within budget" if GO else f"OVER the {MAX_HOURS}h guard, not starting.")

## Stage 4. The run

In [ ]:
assert LEAK_OK and CLEAN, "leak checks failed"
assert ENCODER_MATCH, "encoder does not match 13, this would not be one variable"
assert BLOCK_OK, "the ratio block is not a pure function of the features"
assert GO, "over the time guard"
if not SMOKE:
    assert ALIGNED, "fold sha mismatch"

pre = "SMOKE_" if SMOKE else ""
oof = {mb: np.zeros(len(train)) for mb in MAX_BINS}
tst = {mb: np.zeros((5, len(test))) for mb in MAX_BINS}
per_fold = {mb: [] for mb in MAX_BINS}

t_start = time.time()
for f in range(5):
    tr, va, Xtr, Xva, Xte = build_arm(f, want_test=True)
    line = []
    for mb in MAX_BINS:
        m = make_xgb(N_EST, mb)
        m.fit(Xtr, y[tr])
        oof[mb][va] = m.predict_proba(Xva)[:, 1]
        tst[mb][f] = m.predict_proba(Xte)[:, 1]
        a = float(roc_auc_score(y[va], oof[mb][va]))
        per_fold[mb].append(a)
        line.append(f"{mb}:{a:.6f}")
        del m
        gc.collect()
    del Xtr, Xva, Xte
    gc.collect()
    for mb in MAX_BINS:
        np.save(OUT / f"{pre}PARTIAL_xgb_bin{mb}_oof.npy", oof[mb])
        np.save(OUT / f"{pre}PARTIAL_xgb_bin{mb}_test.npy", tst[mb])
    done = time.time() - t_start
    print(f"fold {f}  " + "  ".join(line))
    print(f"         elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")

print(f"\nall folds done in {hhmm(time.time() - t_start)}")

In [ ]:
cv = {mb: float(np.mean(per_fold[mb])) for mb in MAX_BINS}
sd = {mb: float(np.std(per_fold[mb])) for mb in MAX_BINS}
base = np.array(per_fold[BASE_BIN])

print("The curve. max_bin=256 is row 93 refit, not a treatment.\n")
print(f"{'max_bin':>9} {'CV':>10} {'sd':>9} {'vs 256':>11} {'folds':>8} {'t(4)':>8}")
for mb in MAX_BINS:
    d = np.array(per_fold[mb]) - base
    if mb == BASE_BIN:
        print(f"{mb:>9} {cv[mb]:10.6f} {sd[mb]:9.6f}")
        continue
    s = d.std(ddof=1)
    t = d.mean() / (s / np.sqrt(5)) if s > 0 else float("nan")
    print(f"{mb:>9} {cv[mb]:10.6f} {sd[mb]:9.6f} {d.mean():+11.6f} "
          f"{int((d > 0).sum()):6d}/5 {t:8.2f}")

repro = cv[BASE_BIN] - BASELINE_CV
print(f"\nreproduction of {BASELINE_ROW}: {cv[BASE_BIN]:.6f} vs {BASELINE_CV:.6f}"
      f"  delta {repro:+.2e}")
if SMOKE:
    print("SMOKE: this did NOT run at full size, so the reproduction check DID NOT RUN.")
else:
    print("REPRODUCED" if abs(repro) < 1e-4 else "FAILED - every number above is void")

best = max(MAX_BINS, key=lambda mb: cv[mb])
d = np.array(per_fold[best]) - base
print(f"\nbest max_bin {best} at {cv[best]:.6f}, paired {d.mean():+.6f}, "
      f"{int((d > 0).sum())}/5 folds")
print(f"largest distinct count in the data: {MAX_DISTINCT:,}")
print(f"kito_pl's rule predicts the curve stops improving once max_bin passes that.")
above = [mb for mb in MAX_BINS if mb >= MAX_DISTINCT]
if len(above) >= 2:
    flat = cv[above[-1]] - cv[above[0]]
    print(f"  across the arms at or above it ({above}): {flat:+.6f}")
    print(f"  {'FLAT, the rule holds' if abs(flat) < 0.0002 else 'still moving, the rule does not hold here'}")
print(f"\nfor context: our best single model was row 93 at {BASELINE_CV:.6f},")
print(f"  the best public XGBoost+TE on this split is 0.968372,")
print(f"  and the estimated out-of-fold ceiling is about 0.97006.")

In [ ]:
for mb in MAX_BINS:
    np.save(OUT / f"{pre}xgb_bin{mb}_oof.npy", oof[mb])
    np.save(OUT / f"{pre}xgb_bin{mb}_test.npy", tst[mb].mean(axis=0))
    print(f"wrote {pre}xgb_bin{mb}_oof.npy, {pre}xgb_bin{mb}_test.npy")

print("\nledger lines:")
for mb in MAX_BINS:
    print(f"  name    xgb_bin{mb}\n  cv_mean {cv[mb]:.6f}\n  cv_std  {sd[mb]:.6f}")
print(f"\n  leak checks {'PASS' if (LEAK_OK and CLEAN) else 'FAILED'}, "
      f"encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is a separate notebook and a separate ledger row.")